In [0]:
# COMMAND ----------
# Gold Layer Notebook
# Purpose: Build Gold layer tables (dim_member, dim_provider, fact_claims, hospital_summary_enhanced)
# Inputs: Silver.health_claims
# Outputs: Gold tables
# Author: <Your Name>
# Date: <YYYY-MM-DD>
# Notes: Includes error handling, logging, schema validation, and performance optimizations

# COMMAND ----------
# 1. Imports & Config
import traceback
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

gold_path = "abfss://gold@mvnazureproject.dfs.core.windows.net"

# COMMAND ----------
# 2. Ensure log schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS mvn_db_catalog.logs")

# Explicit schema for logging
log_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("step", StringType(), True),
    StructField("status", StringType(), True),
    StructField("error", StringType(), True)
])

def log_etl(step, status, error=None):
    log_df = spark.createDataFrame(
        [(datetime.now(), step, status, str(error) if error else None)],
        schema=log_schema
    )
    log_df.write.format("delta").mode("append").saveAsTable("mvn_db_catalog.logs.etl_runs")

# COMMAND ----------
# 3. Load Silver & Validate Schema
expected_cols = {"ClaimID","MemberID","ProviderID","Hospital","ClaimAmount","ClaimDate","Status"}
df_silver = spark.table("mvn_db_catalog.silver.health_claims")

missing = expected_cols - set(df_silver.columns)
if missing:
    raise ValueError(f"❌ Missing columns in Silver: {missing}")

# Deduplicate for safety
df_silver = df_silver.dropDuplicates(["ClaimID"])

# COMMAND ----------
# 4. Update dim_member
try:
    df_silver.select("MemberID").distinct() \
        .write.format("delta").mode("overwrite") \
        .option("path", f"{gold_path}/dim_member/") \
        .saveAsTable("mvn_db_catalog.gold.dim_member")

    # Trigger action inside try
    dim_count = spark.table("mvn_db_catalog.gold.dim_member").count()
    print("✅ dim_member updated:", dim_count)
    log_etl("dim_member","SUCCESS")
except Exception as e:
    print("❌ Error updating dim_member:", str(e))
    traceback.print_exc()
    log_etl("dim_member","FAILED",str(e))
    raise

# COMMAND ----------
# 5. Update dim_provider
try:
    df_silver.select("ProviderID","Hospital").distinct() \
        .write.format("delta").mode("overwrite") \
        .option("path", f"{gold_path}/dim_provider/") \
        .saveAsTable("mvn_db_catalog.gold.dim_provider")

    provider_count = spark.table("mvn_db_catalog.gold.dim_provider").count()
    print("✅ dim_provider updated:", provider_count)
    log_etl("dim_provider","SUCCESS")
except Exception as e:
    print("❌ Error updating dim_provider:", str(e))
    traceback.print_exc()
    log_etl("dim_provider","FAILED",str(e))
    raise

# COMMAND ----------
# 6. SMART UPSERT (MERGE) FOR fact_claims
try:
    if not spark.catalog.tableExists("mvn_db_catalog.gold.fact_claims"):
        df_silver.write.format("delta").mode("overwrite") \
            .option("path", f"{gold_path}/fact_claims/") \
            .saveAsTable("mvn_db_catalog.gold.fact_claims")
        print("✅ fact_claims created fresh")
    else:
        df_silver.createOrReplaceTempView("incoming_silver_batch")
        spark.sql("""
            MERGE INTO mvn_db_catalog.gold.fact_claims AS target
            USING incoming_silver_batch AS source
            ON target.ClaimID = source.ClaimID
            WHEN MATCHED THEN UPDATE SET
              target.MemberID = source.MemberID,
              target.ProviderID = source.ProviderID,
              target.Hospital = source.Hospital,
              target.ClaimAmount = source.ClaimAmount,
              target.ClaimDate = source.ClaimDate,
              target.Status = source.Status
            WHEN NOT MATCHED THEN INSERT *
        """)
        print("✅ fact_claims merged successfully")

    fact_count = spark.table("mvn_db_catalog.gold.fact_claims").count()
    print("fact_claims row count:", fact_count)
    log_etl("fact_claims","SUCCESS")
except Exception as e:
    print("❌ Error updating fact_claims:", str(e))
    traceback.print_exc()
    log_etl("fact_claims","FAILED",str(e))
    raise

# COMMAND ----------
# 7. Refresh hospital_summary_enhanced
try:
    df_summary = spark.sql("""
        SELECT Hospital,
               COUNT(ClaimID) as TotalClaims,
               SUM(ClaimAmount) as TotalGrossAmount,
               SUM(CASE WHEN Status='Approved' THEN ClaimAmount ELSE 0 END) as ApprovedAmount,
               SUM(CASE WHEN Status='Rejected' THEN ClaimAmount ELSE 0 END) as RejectedAmount,
               SUM(CASE WHEN Status='Pending'  THEN ClaimAmount ELSE 0 END) as PendingAmount
        FROM mvn_db_catalog.gold.fact_claims
        GROUP BY Hospital
    """)
    df_summary.write.format("delta").mode("overwrite") \
        .option("path", f"{gold_path}/hospital_summary_enhanced/") \
        .saveAsTable("mvn_db_catalog.gold.hospital_summary_enhanced")

    summary_count = spark.table("mvn_db_catalog.gold.hospital_summary_enhanced").count()
    print("✅ hospital_summary_enhanced refreshed:", summary_count)
    log_etl("hospital_summary_enhanced","SUCCESS")
except Exception as e:
    print("❌ Error refreshing hospital_summary_enhanced:", str(e))
    traceback.print_exc()
    log_etl("hospital_summary_enhanced","FAILED",str(e))
    raise

# COMMAND ----------
print("🎉 Gold layer pipeline completed successfully")

# COMMAND ----------
# 8. Performance Maintenance (Optional, run periodically)
# Compact small files and clean old versions
# spark.sql("OPTIMIZE mvn_db_catalog.gold.fact_claims ZORDER BY (ClaimDate)")
# spark.sql("VACUUM mvn_db_catalog.gold.fact_claims RETAIN 168 HOURS")